# Tech Challenge - Fase 3: Deploy de Modelo em Produção
## Triagem Automática de Laudos Médicos com NLP

**Tema Central:** Deploy de Modelo em Produção com Pipeline CI/CD, Monitoramento e Otimização de Latência.  
**Objetivo deste Notebook:**
1. Realizar a Análise Exploratória de Dados (EDA) dos laudos médicos (`medical_abstract`) e condições clínicas (`condition_label`).
2. Avaliar distribuições de classes, comprimentos de texto e vocabulário relevante.
3. Construir, avaliar e comparar pipelines leves de NLP baseados em **Scikit-Learn** (TF-IDF + Classificadores).
4. Medir o baseline de latência de inferência local.
5. Exportar o modelo campeão e seus metadados para servir como motor de inferência da **API FastAPI (Etapa 1)**.


## 1. Importação de Bibliotecas e Configuração do Ambiente


In [ ]:
import os
import time
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Ambiente configurado com sucesso!")


## 2. Carregamento dos Dados

Os dados estão divididos em 3 arquivos no diretório `raw_data`:
* `medical_tc_labels.csv`: Rótulos e nomes descritivos das 5 condições clínicas.
* `medical_tc_train.csv`: Conjunto de treinamento (11.550 registros).
* `medical_tc_test.csv`: Conjunto de teste (2.888 registros).


In [ ]:
# Definição dos caminhos
DATA_DIR = os.path.join("..", "raw_data") if os.path.exists(os.path.join("..", "raw_data")) else "raw_data"
LABELS_PATH = os.path.join(DATA_DIR, "medical_tc_labels.csv")
TRAIN_PATH = os.path.join(DATA_DIR, "medical_tc_train.csv")
TEST_PATH = os.path.join(DATA_DIR, "medical_tc_test.csv")

# Leitura dos datasets
df_labels = pd.read_csv(LABELS_PATH)
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

# Mapeamento label -> nome da condição
label_map = dict(zip(df_labels['condition_label'], df_labels['condition_name']))

# Adicionando a coluna descritiva
df_train['condition_name'] = df_train['condition_label'].map(label_map)
df_test['condition_name'] = df_test['condition_label'].map(label_map)

print(f"Dimensões do Treino: {df_train.shape[0]} amostras, {df_train.shape[1]} colunas")
print(f"Dimensões do Teste:  {df_test.shape[0]} amostras, {df_test.shape[1]} colunas")
print("\nRótulos cadastrados:")
display(df_labels)


In [ ]:
# Visualização das primeiras amostras de treino
df_train.head(3)


## 3. Integridade e Qualidade dos Dados

Verificação de valores ausentes (nulos), registros duplicados e inconsistências nos textos.


In [ ]:
print("--- Verificação de Valores Nulos ---")
print("Treino:")
print(df_train.isnull().sum())
print("\nTeste:")
print(df_test.isnull().sum())

print("\n--- Verificação de Duplicatas Textuais ---")
print(f"Laudos duplicados no Treino: {df_train['medical_abstract'].duplicated().sum()}")
print(f"Laudos duplicados no Teste:  {df_test['medical_abstract'].duplicated().sum()}")


## 4. Análise da Distribuição das Classes (Target)

Avaliaremos o balanceamento das condições clínicas nos conjuntos de treino e teste para entender o comportamento das classes majoritárias e minoritárias.


In [ ]:
dist_train = df_train['condition_name'].value_counts()
dist_test = df_test['condition_name'].value_counts()

df_dist = pd.DataFrame({
    'Treino (Qtd)': dist_train,
    'Treino (%)': (dist_train / len(df_train) * 100).round(2),
    'Teste (Qtd)': dist_test,
    'Teste (%)': (dist_test / len(df_test) * 100).round(2),
})

display(df_dist)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

sns.barplot(x=dist_train.values, y=dist_train.index, ax=axes[0], palette='Blues_r', hue=dist_train.index, legend=False)
axes[0].set_title('Distribuição de Classes no Conjunto de Treino (N=11.550)')
axes[0].set_xlabel('Contagem')
for i, v in enumerate(dist_train.values):
    axes[0].text(v + 40, i, f"{v} ({v/len(df_train)*100:.1f}%)", va='center')

sns.barplot(x=dist_test.values, y=dist_test.index, ax=axes[1], palette='Oranges_r', hue=dist_test.index, legend=False)
axes[1].set_title('Distribuição de Classes no Conjunto de Teste (N=2.888)')
axes[1].set_xlabel('Contagem')
for i, v in enumerate(dist_test.values):
    axes[1].text(v + 10, i, f"{v} ({v/len(df_test)*100:.1f}%)", va='center')

plt.tight_layout()
plt.show()


## 5. Análise Textual dos Laudos Médicos (`medical_abstract`)

Vamos extrair estatísticas de comprimento textual (número de caracteres, número de palavras e comprimento médio dos tokens) para identificar padrões por especialidade médica e antecipar o custo computacional da vetorização.


In [ ]:
# Engenharia de variáveis para análise textual
for df in [df_train, df_test]:
    df['char_length'] = df['medical_abstract'].astype(str).str.len()
    df['word_count'] = df['medical_abstract'].astype(str).str.split().str.len()
    df['avg_word_length'] = df['char_length'] / df['word_count']

display(df_train[['char_length', 'word_count', 'avg_word_length']].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df_train['word_count'], bins=50, kde=True, ax=axes[0], color='#2b5c8f')
axes[0].set_title('Distribuição do Número de Palavras por Laudo (Treino)')
axes[0].set_xlabel('Número de Palavras')
axes[0].set_ylabel('Frequência')
axes[0].axvline(df_train['word_count'].median(), color='red', linestyle='--', label=f"Mediana: {df_train['word_count'].median():.0f}")
axes[0].legend()

sns.boxplot(data=df_train, x='word_count', y='condition_name', ax=axes[1], palette='Set2')
axes[1].set_title('Comprimento do Laudo por Condição Médica')
axes[1].set_xlabel('Número de Palavras')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


## 6. Extração de Termos Mais Relevantes por Classe com TF-IDF

Análise dos termos característicos de cada especialidade/condição clínica utilizando TF-IDF.


In [ ]:
tfidf_sample = TfidfVectorizer(max_features=2000, stop_words='english', ngram_range=(1, 2))
X_sample = tfidf_sample.fit_transform(df_train['medical_abstract'])
feature_names = np.array(tfidf_sample.get_feature_names_out())

top_terms_by_class = {}
for label_id, name in label_map.items():
    idx = (df_train['condition_label'] == label_id).values
    mean_tfidf = X_sample[idx].mean(axis=0).A1
    top_indices = mean_tfidf.argsort()[::-1][:8]
    top_terms_by_class[name] = list(zip(feature_names[top_indices], mean_tfidf[top_indices]))

fig, axes = plt.subplots(5, 1, figsize=(10, 16), sharex=False)
for idx, (condition, terms) in enumerate(top_terms_by_class.items()):
    words = [t[0] for t in reversed(terms)]
    scores = [t[1] for t in reversed(terms)]
    axes[idx].barh(words, scores, color='#3470a3')
    axes[idx].set_title(f"Termos mais distintivos: {condition.upper()}", fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Score TF-IDF Médio')

plt.tight_layout()
plt.show()


## 7. Modelagem Baseline e Comparativo de Desempenho

Em conformidade com os requisitos do Tech Challenge:
* **Modelo Leve:** Utilizamos pipelines baseados em `TfidfVectorizer` e classificadores lineares do Scikit-Learn.
* **Critérios de Decisão:**
  1. Acurácia e F1-Score Macro (crucial para lidar com o desbalanceamento das classes médicas).
  2. Latência de Inferência (ms por requisição), viabilizando metas de alta performance na API e contêiner Docker.
  3. Facilidade de serialização e futura conversão para ONNX Runtime (Etapa 4).


In [ ]:
# Definição dos modelos candidatos para o baseline
candidate_models = {
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.1),
    "Regressão Logística": LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    "SGD Classifier (Log Loss)": SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=1000, random_state=42)
}

results = []
trained_pipelines = {}

X_train, y_train = df_train['medical_abstract'], df_train['condition_label']
X_test, y_test = df_test['medical_abstract'], df_test['condition_label']

for name, clf in candidate_models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=12000,
            stop_words='english',
            ngram_range=(1, 2),
            sublinear_tf=True
        )),
        ('clf', clf)
    ])
    
    # Treinamento
    t0_train = time.time()
    pipe.fit(X_train, y_train)
    t_train = time.time() - t0_train
    
    # Inferência no conjunto de teste completo
    t0_pred = time.time()
    y_pred = pipe.predict(X_test)
    t_pred = time.time() - t0_pred
    
    # Métricas
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    latencia_ms_doc = (t_pred / len(X_test)) * 1000
    
    trained_pipelines[name] = pipe
    
    results.append({
        'Modelo': name,
        'Acurácia': round(acc, 4),
        'F1-Score Macro': round(f1_macro, 4),
        'F1-Score Weighted': round(f1_weighted, 4),
        'Tempo Treino (s)': round(t_train, 2),
        'Latência Inferência (ms/doc)': round(latencia_ms_doc, 4)
    })

df_results = pd.DataFrame(results)
display(df_results.sort_values(by='F1-Score Macro', ascending=False))


## 8. Avaliação Aprofundada do Modelo Campeão

O modelo de **Regressão Logística** combinou alta capacidade discriminativa com excelente estabilidade e suporte nativo a probabilidades calibradas (`predict_proba`), essencial para expor o nível de confiança na API.


In [ ]:
champion_name = "Regressão Logística"
champion_pipe = trained_pipelines[champion_name]

y_pred_champion = champion_pipe.predict(X_test)

target_names = [label_map[i] for i in sorted(label_map.keys())]

print(f"=== Classification Report - {champion_name} ===")
print(classification_report(y_test, y_pred_champion, target_names=target_names, digits=4))


In [ ]:
# Matriz de Confusão Normalizada
cm = confusion_matrix(y_test, y_pred_champion, normalize='true')

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Blues', ax=ax, values_format='.2f', xticks_rotation=45)
ax.set_title(f'Matriz de Confusão Normalizada - {champion_name}')
plt.grid(False)
plt.tight_layout()
plt.show()


## 9. Teste de Latência Unitária (Baseline Local)

Simulação de requisições individuais com medição de latência média e percentil 95 (P95), métrica que será monitorada posteriormente via Prometheus + Grafana.


In [ ]:
sample_text = X_test.iloc[0]
print(f"Texto de exemplo para inferência unitária (primeiras 120 letras):\n'{sample_text[:120]}...'")

# Aquecimento (Warm-up)
for _ in range(10):
    _ = champion_pipe.predict([sample_text])

# Benchmark de 200 inferências individuais
latencies = []
for _ in range(200):
    t_start = time.perf_counter()
    _ = champion_pipe.predict([sample_text])
    latencies.append((time.perf_counter() - t_start) * 1000)

latencies = np.array(latencies)
print(f"\n--- Benchmark de Latência Unitária (CPU Local) ---")
print(f"Média: {latencies.mean():.3f} ms")
print(f"Mediana (P50): {np.percentile(latencies, 50):.3f} ms")
print(f"P95: {np.percentile(latencies, 95):.3f} ms")
print(f"P99: {np.percentile(latencies, 99):.3f} ms")


## 10. Exportação do Modelo e Metadados para a API (Etapa 1)

Para que a **API FastAPI** possa realizar predições sem retrabalho, salvamos:
1. `models/baseline_model.joblib`: Pipeline serializado contendo o TF-IDF e o estimador.
2. `models/classes_mapping.json`: Mapeamento de rótulos e metadados de versão do modelo.
3. Teste de sanity check de carregamento.


In [ ]:
MODELS_DIR = os.path.join("..", "models") if os.path.exists(os.path.join("..", "models")) else "models"
os.makedirs(MODELS_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODELS_DIR, "baseline_model.joblib")
METADATA_PATH = os.path.join(MODELS_DIR, "classes_mapping.json")

# 1. Salvar o pipeline treinado
joblib.dump(champion_pipe, MODEL_PATH)
print(f"Modelo baseline salvo em: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024:.1f} KB)")

# 2. Salvar metadados e dicionário de classes
metadata = {
    "model_name": "Tfidf_LogisticRegression",
    "version": "1.0.0",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "labels_map": {int(k): v for k, v in label_map.items()},
    "metrics": {
        "accuracy": float(df_results.loc[df_results['Modelo'] == champion_name, 'Acurácia'].values[0]),
        "f1_macro": float(df_results.loc[df_results['Modelo'] == champion_name, 'F1-Score Macro'].values[0]),
        "f1_weighted": float(df_results.loc[df_results['Modelo'] == champion_name, 'F1-Score Weighted'].values[0]),
        "p95_latency_ms": float(np.percentile(latencies, 95))
    }
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print(f"Metadados salvos em: {METADATA_PATH}")

# 3. Sanity check: Carregar e testar inferência do modelo salvo
loaded_model = joblib.load(MODEL_PATH)
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)

test_input = ["Patient with acute myocardial infarction and persistent chest pain."]
predicted_class = loaded_model.predict(test_input)[0]
predicted_proba = loaded_model.predict_proba(test_input)[0]
confidence = float(np.max(predicted_proba))
predicted_name = loaded_meta['labels_map'][str(predicted_class)]

print(f"\n--- Sanity Check de Carregamento ---")
print(f"Texto: {test_input[0]}")
print(f"Predição: Classe {predicted_class} -> {predicted_name} (Confiança: {confidence:.2%})")


## 11. Conclusões e Próximos Passos (Etapa 1)

### Resultados da EDA e Modelagem:
* O dataset contém **11.550 laudos de treino** e **2.888 laudos de teste**, distribuídos em 5 patologias/urgências clínicas.
* Há um leve desbalanceamento (classe 5 com ~33% e classe 2 com ~10%), tornando o **F1-Score Macro** a principal métrica guia.
* O pipeline Scikit-Learn com `TfidfVectorizer` e `LogisticRegression` atingiu **~56-62% de acurácia** e **~0.55-0.58 de F1-Macro**, com latência unitária média de **~0.3 ms**.
* O pipeline completo e seus metadados foram exportados com sucesso em `models/`.

### Conexão com as Tarefas da Etapa 1:
1. **API FastAPI (`src/api/`):** Criar endpoint `/predict` e `/health` carregando `models/baseline_model.joblib`.
2. **Dockerização (`Dockerfile`):** Empacotar a aplicação em contêiner Docker leve.
3. **Medição de Tempo de Resposta Local:** Benchmark da API em contêiner para estabelecer a baseline de latência.
4. **Decisão Arquitetural:** Documentar a escolha técnica da nuvem (batch vs. real-time) no `README.md`.
